In [1]:
import pandas as pd
import numpy as np

# Generate synthetic sales data
np.random.seed(42)
dates = pd.date_range(start="2025-01-01", end="2025-12-31", freq="D")
products = ["Laptop", "Smartphone", "Tablet", "Monitor", "Headphones"]
regions = ["North", "East", "South", "West"]

data = {
    "Date": np.random.choice(dates, size=500),
    "Product": np.random.choice(products, size=500),
    "Region": np.random.choice(regions, size=500),
    "Sales": np.random.uniform(100, 1500, size=500).round(2),
    "Units_Sold": np.random.randint(1, 10, size=500)
}

df = pd.DataFrame(data)
# Create a Profit column with some mathematical relationship + random noise
df["Profit"] = (df["Sales"] * np.random.uniform(0.15, 0.40, size=500)).round(2)

# Introduce a few deliberate missing values for the cleaning demonstration
df.loc[df.sample(frac=0.02).index, "Sales"] = np.nan

df.to_csv("sales_data.csv", index=False)
print("Synthetic 'sales_data.csv' created successfully!")

Synthetic 'sales_data.csv' created successfully!


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

class SalesDataAnalyzer:
    """
    A comprehensive Sales Data Analysis and Visualization tool 
    encapsulating Pandas, NumPy, Matplotlib, and Seaborn workflows.
    """
    
    def __init__(self, file_path=None):
        """Constructor to initialize and optionally load data."""
        self.data = None
        if file_path:
            self.load_data(file_path)
            
    def __del__(self):
        """Destructor to clean up resources if needed."""
        # Clean up any open matplotlib plots to free up memory
        plt.close('all')
        print("[Destructor]: Resources cleared and memory freed.")

    def load_data(self, file_path):
        """Load data from a CSV file with robust exception handling."""
        try:
            if not os.path.exists(file_path):
                raise FileNotFoundError(f"The file '{file_path}' does not exist.")
            
            self.data = pd.read_csv(file_path)
            # Ensure Date column is in datetime format
            if 'Date' in self.data.columns:
                self.data['Date'] = pd.to_datetime(self.data['Date'])
            print(f"🎉 Successfully loaded data from {file_path}. Shape: {self.data.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            self.data = None

    def explore_data(self):
        """Display basic structural information about the dataset."""
        if self.data is None:
            print("⚠️ No data loaded. Please load data first.")
            return
        
        print("\n=== Data Head ===")
        print(self.data.head())
        print("\n=== Data Info ===")
        print(self.data.info())
        print("\n=== Data Description ===")
        print(self.data.describe(include='all'))

    def clean_data(self):
        """Handle missing values and verify column structures."""
        if self.data is None: return
        
        print("\n--- Missing Values Before Cleaning ---")
        print(self.data.isnull().sum())
        
        # Strategy: Fill numeric null values with the median of that column
        numeric_cols = self.data.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if self.data[col].isnull().any():
                median_val = self.data[col].median()
                self.data[col] = self.data[col].fillna(median_val)
                print(f"Filled missing values in '{col}' with median: {median_val}")
                
        # Fill categorical/object columns with 'Unknown'
        categorical_cols = self.data.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if self.data[col].isnull().any():
                self.data[col] = self.data[col].fillna('Unknown')
                print(f"Filled missing values in '{col}' with 'Unknown'")
                
        print("✅ Data cleaning complete.")

    def mathematical_operations(self):
        """Requirement A & B: Demonstrate NumPy array conversion, slicing, and element-wise operations."""
        if self.data is None: return
        
        print("\n=== NumPy Array Operations ===")
        # Convert DataFrame columns to NumPy arrays
        sales_array = self.data['Sales'].to_numpy()
        units_array = self.data['Units_Sold'].to_numpy()
        
        # Slicing demonstration
        print(f"First 5 elements of Sales Array: {sales_array[:5]}")
        print(f"Sliced subset (indices 10 to 15): {sales_array[10:15]}")
        
        # Element-wise operations: Calculate average price per unit across the array
        avg_unit_price = sales_array / np.where(units_array == 0, 1, units_array) # Avoid division by zero
        print(f"Calculated Price-per-Unit (First 5 element-wise evaluations):\n{avg_unit_price[:5]}")
        
        # Mutating DataFrame back using this array operation result
        self.data['Price_Per_Unit'] = np.round(avg_unit_price, 2)
        print("Added computed metric 'Price_Per_Unit' back to DataFrame.")

    def combine_data(self, other_dataframe):
        """Combine current DataFrame with another using concatenation."""
        if self.data is None: return
        print("\n=== Combining DataFrames ===")
        self.data = pd.concat([self.data, other_dataframe], ignore_index=True)
        print(f"Data combined successfully. New Shape: {self.data.shape}")

    def split_data(self):
        """Split data into multiple DataFrames based on standard criteria (e.g., Region)."""
        if self.data is None: return {}
        print("\n=== Splitting Data by Region ===")
        regions = self.data['Region'].unique()
        split_dict = {region: self.data[self.data['Region'] == region].copy() for region in regions}
        for region, df_sub in split_dict.items():
            print(f" - Sub-dataset created for Region: '{region}' ({len(df_sub)} rows)")
        return split_dict

    def search_sort_filter(self):
        """Implement explicit interactive search, sort, and filtering operations."""
        if self.data is None: return
        print("\n=== Search, Sort & Filter Demo ===")
        
        # Filter data for a specific criterion
        target_region = "North"
        filtered_df = self.data[self.data['Region'] == target_region]
        print(f"💡 Filtered Data (Region == '{target_region}'): Found {len(filtered_df)} rows.")
        
        # Sort data
        sorted_df = filtered_df.sort_values(by='Sales', ascending=False)
        print(f"🔝 Top 3 Sales records within '{target_region}' Region:")
        print(sorted_df[['Date', 'Product', 'Sales', 'Profit']].head(3))
        
        # Search via query condition (e.g., Sales greater than $1400)
        high_value_sales = self.data[self.data['Sales'] > 1400]
        print(f"🔍 Search Result: Found {len(high_value_sales)} high-value orders (> $1400).")

    def aggregate_functions(self):
        """Apply pandas-native group aggregation rules."""
        if self.data is None: return
        print("\n=== Aggregation Metrics (By Product) ===")
        agg_res = self.data.groupby('Product').agg({
            'Sales': ['sum', 'mean'],
            'Units_Sold': 'sum',
            'Profit': 'sum'
        })
        print(agg_res)

    def statistical_analysis(self):
        """Requirement D: Calculate explicit statistical distributions via NumPy/Pandas functions."""
        if self.data is None: return
        print("\n=== Statistical Variance & Distribution metrics ===")
        sales = self.data['Sales']
        
        print(f"Standard Deviation of Sales : {sales.std():.2f}")
        print(f"Variance of Sales           : {sales.var():.2f}")
        print(f"25th Percentile (Q1)        : {sales.quantile(0.25):.2f}")
        print(f"50th Percentile (Median)    : {sales.quantile(0.50):.2f}")
        print(f"75th Percentile (Q3)        : {sales.quantile(0.75):.2f}")

    def create_pivot_table(self):
        """Generate interactive structural multidimensional pivot matrices."""
        if self.data is None: return
        print("\n=== Regional Product Performance Pivot Matrix (Total Profit) ===")
        pivot = pd.pivot_table(
            self.data, 
            values='Profit', 
            index='Product', 
            columns='Region', 
            aggfunc='sum', 
            fill_value=0
        )
        print(pivot)
        return pivot

    def visualize_data(self):
        """Requirement E & F: Generates customized visualization charts via Matplotlib & Seaborn."""
        if self.data is None:
            print("⚠️ No data available to visualize.")
            return
        
        print("\n📊 Generating and exporting visualization suite plots...")
        os.makedirs('output_plots', exist_ok=True)
        
        # --- 1. Matplotlib Subplot Figure (Line, Scatter, Bar, Pie, Stack Plot) ---
        fig, axs = plt.subplots(3, 2, figsize=(16, 18))
        fig.suptitle('Comprehensive Corporate Sales Visual Dashboard', fontsize=20, weight='bold')
        
        # Data extractions for plots
        prod_summary = self.data.groupby('Product').sum(numeric_only=True).reset_index()
        region_summary = self.data.groupby('Region').sum(numeric_only=True).reset_index()
        monthly_sales = self.data.set_index('Date').resample('ME').sum(numeric_only=True).reset_index()
        
        # A. Line Plot: Sales trend over time
        axs[0, 0].plot(monthly_sales['Date'], monthly_sales['Sales'], marker='o', color='tab:blue', linewidth=2)
        axs[0, 0].set_title('Monthly Total Revenue Performance Trends')
        axs[0, 0].tick_params(axis='x', rotation=30)
        axs[0, 0].grid(True, linestyle='--')
        
        # B. Bar Chart: Product breakdown
        axs[0, 1].bar(prod_summary['Product'], prod_summary['Sales'], color='tab:green', edgecolor='black')
        axs[0, 1].set_title('Product-wise Gross Revenue Analysis')
        
        # C. Scatter Plot: Sales vs Profit
        axs[1, 0].scatter(self.data['Sales'], self.data['Profit'], alpha=0.6, color='tab:orange', edgecolors='w')
        axs[1, 0].set_title('Transaction Distribution: Sales Volatility vs Profit Margin')
        axs[1, 0].set_xlabel('Sales Revenue')
        axs[1, 0].set_ylabel('Net Profit margin')
        
        # D. Pie Chart: Region distribution
        axs[1, 1].pie(region_summary['Sales'], labels=region_summary['Region'], autopct='%1.1f%%', startangle=140, colors=plt.cm.Paired.colors)
        axs[1, 1].set_title('Market Share Contribution ratio by Region')
        
        # E. Histogram: Distribution of individual sales values
        axs[2, 0].hist(self.data['Sales'], bins=20, color='purple', edgecolor='black', alpha=0.7)
        axs[2, 0].set_title('Frequency Distribution Profile of Unit Invoices')
        
        # F. Stack Plot Chart: Accumulated Performance Simulation over standard units
        sample_stack = self.data.head(10)
        axs[2, 1].stackplot(range(len(sample_stack)), sample_stack['Sales'], sample_stack['Profit'], labels=['Gross Sales', 'Net Margin Profit'], colors=['skyblue', 'salmon'])
        axs[2, 1].set_title('Invoice Cumulative Area Breakdown Profile (First 10 records)')
        axs[2, 1].legend(loc='upper left')
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        matplotlib_output = 'output_plots/matplotlib_dashboard.png'
        plt.savefig(matplotlib_output, dpi=200)
        plt.close()
        print(f" Saved structural Matplotlib composite matrix to: {matplotlib_output}")

        # --- 2. Seaborn Enhanced Plots ---
        # Plot A: Box plot profile
        plt.figure(figsize=(10, 6))
        sns.set_theme(style="whitegrid")
        sns.boxplot(x='Product', y='Sales', hue='Region', data=self.data)
        plt.title('Product Price Distribution Spread across Regions (Outlier Mapping Matrix)')
        seaborn_box_output = 'output_plots/seaborn_boxplot.png'
        plt.savefig(seaborn_box_output, dpi=200)
        plt.close()
        print(f" Saved Seaborn Boxplot evaluation context to: {seaborn_box_output}")

        # Plot B: Heatmap representation of pivot correlation matrix
        plt.figure(figsize=(8, 6))
        pivot_data = self.create_pivot_table()
        sns.heatmap(pivot_data, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=.5)
        plt.title('Heatmap Matrix: Profit Concentrations by Product & Region Segment')
        seaborn_heat_output = 'output_plots/seaborn_heatmap.png'
        plt.savefig(seaborn_heat_output, dpi=200)
        plt.close()1
        print(f" Saved Seaborn Interactive Heatmap Matrix to: {seaborn_heat_output}")


# --- USER INTERFACE (UI) & RUNTIME FLOW ENGINE ---
def main_menu():
    analyzer = None
    default_filename = "sales_data.csv"
    
    while True:
        print("\n" + "="*50)
        print("📊 PANDAS ANALYZER & DATA VISUALIZATION ENGINE 📊")
        print("="*50)
        print("1. Initialize Engine & Load Sales Dataset")
        print("2. Explore Dataset Structures (head, info, summary)")
        print("3. Execute Data Cleaning Strategy (Handle Missing Values)")
        print("4. Perform Array Mathematics & Transformations (NumPy Engine)")
        print("5. Search, Filter, & Sort Sub-segments")
        print("6. Run Core Dataset Statistical & Aggregation Summaries")
        print("7. Build Dynamic Pivot Table View")
        print("8. Generate & Export Dashboard Visualizations (Matplotlib & Seaborn)")
        print("9. Quit Program")
        print("="*50)
        
        choice = input("Select an analytical operation option (1-9): ").strip()
        
        if choice == '1':
            filepath = input(f"Enter CSV file path [Press Enter to use default '{default_filename}']: ").strip()
            if not filepath:
                filepath = default_filename
            analyzer = SalesDataAnalyzer(filepath)
            
        elif choice in ['2', '3', '4', '5', '6', '7', '8']:
            if analyzer is None or analyzer.data is None:
                print("⚠️ [Access Denied]: Initialize the Analyzer state and load data first using option 1.")
                continue
                
            if choice == '2':
                analyzer.explore_data()
            elif choice == '3':
                analyzer.clean_data()
            elif choice == '4':
                analyzer.mathematical_operations()
                # Demonstration of functional splitting criteria layout requirement
                analyzer.split_data()
            elif choice == '5':
                analyzer.search_sort_filter()
            elif choice == '6':
                analyzer.aggregate_functions()
                analyzer.statistical_analysis()
            elif choice == '7':
                analyzer.create_pivot_table()
            elif choice == '8':
                analyzer.visualize_data()
                
        elif choice == '9':
            print("\nShutting down engine pipeline. Goodbye!")
            if analyzer:
                del analyzer  # Triggers destructor explicitly
            break
        else:
            print("❌ Invalid entry value string. Input numbers 1 through 9.")

if __name__ == "__main__":
    main_menu()


📊 PANDAS ANALYZER & DATA VISUALIZATION ENGINE 📊
1. Initialize Engine & Load Sales Dataset
2. Explore Dataset Structures (head, info, summary)
3. Execute Data Cleaning Strategy (Handle Missing Values)
4. Perform Array Mathematics & Transformations (NumPy Engine)
5. Search, Filter, & Sort Sub-segments
6. Run Core Dataset Statistical & Aggregation Summaries
7. Build Dynamic Pivot Table View
8. Generate & Export Dashboard Visualizations (Matplotlib & Seaborn)
9. Quit Program


Select an analytical operation option (1-9):  1
Enter CSV file path [Press Enter to use default 'sales_data.csv']:  2


❌ Error loading file: The file '2' does not exist.

📊 PANDAS ANALYZER & DATA VISUALIZATION ENGINE 📊
1. Initialize Engine & Load Sales Dataset
2. Explore Dataset Structures (head, info, summary)
3. Execute Data Cleaning Strategy (Handle Missing Values)
4. Perform Array Mathematics & Transformations (NumPy Engine)
5. Search, Filter, & Sort Sub-segments
6. Run Core Dataset Statistical & Aggregation Summaries
7. Build Dynamic Pivot Table View
8. Generate & Export Dashboard Visualizations (Matplotlib & Seaborn)
9. Quit Program


Select an analytical operation option (1-9):  3


⚠️ [Access Denied]: Initialize the Analyzer state and load data first using option 1.

📊 PANDAS ANALYZER & DATA VISUALIZATION ENGINE 📊
1. Initialize Engine & Load Sales Dataset
2. Explore Dataset Structures (head, info, summary)
3. Execute Data Cleaning Strategy (Handle Missing Values)
4. Perform Array Mathematics & Transformations (NumPy Engine)
5. Search, Filter, & Sort Sub-segments
6. Run Core Dataset Statistical & Aggregation Summaries
7. Build Dynamic Pivot Table View
8. Generate & Export Dashboard Visualizations (Matplotlib & Seaborn)
9. Quit Program


...

⚠️ [Access Denied]: Initialize the Analyzer state and load data first using option 1.

📊 PANDAS ANALYZER & DATA VISUALIZATION ENGINE 📊
1. Initialize Engine & Load Sales Dataset
2. Explore Dataset Structures (head, info, summary)
3. Execute Data Cleaning Strategy (Handle Missing Values)
4. Perform Array Mathematics & Transformations (NumPy Engine)
5. Search, Filter, & Sort Sub-segments
6. Run Core Dataset Statistical & Aggregation Summaries
7. Build Dynamic Pivot Table View
8. Generate & Export Dashboard Visualizations (Matplotlib & Seaborn)
9. Quit Program


Select an analytical operation option (1-9):  4


⚠️ [Access Denied]: Initialize the Analyzer state and load data first using option 1.

📊 PANDAS ANALYZER & DATA VISUALIZATION ENGINE 📊
1. Initialize Engine & Load Sales Dataset
2. Explore Dataset Structures (head, info, summary)
3. Execute Data Cleaning Strategy (Handle Missing Values)
4. Perform Array Mathematics & Transformations (NumPy Engine)
5. Search, Filter, & Sort Sub-segments
6. Run Core Dataset Statistical & Aggregation Summaries
7. Build Dynamic Pivot Table View
8. Generate & Export Dashboard Visualizations (Matplotlib & Seaborn)
9. Quit Program


Select an analytical operation option (1-9):  9



Shutting down engine pipeline. Goodbye!
[Destructor]: Resources cleared and memory freed.
